# Using the Eval Service for Standard Evaluation

Our setting is a standard evaluation where versions are fixed shards, and the agent is replaced by an
**orchestration plan**, each restricted to a subset of the gym's
tools that we generate beforehand and then execute ourselves.

|  | Tutorial (evolving) | This pipeline (standard) |
|---|---|---|
| Who runs the agent loop | **the service** (`react_agent` / `acp_codex_agent`) | **us** |
| Granularity | one agent, all tools | N subagents, each with its own tool allowlist |
| Planning | inside the harness |  an inspectable XML plan |
| Service's role | runs the agent **and** grades | **gym + grader only** |
| Evolving resource | central to the notebook | not used (fixed `(domain, version)`) |
| What we get back | the harness's result | full per-node trace + the same grade |

Grading is **identical** in both: `task.grade()` runs the task's hidden SQL verifiers against the
final database state. That is what makes the two comparable.

> Plan generation, XML parsing and dataset loading live elsewhere in the repo and are **not**
> shown here — the plan below is pasted in already-parsed, exactly as those steps would hand it
> over. Everything in this notebook is the eval service and nothing else.

## Setup

Only the SDK and `openai` are needed.

In [4]:
# pip install:
#   WHEEL=$(curl -fsSL -H "Authorization: Bearer $EVAL_SERVICE_API_KEY" "$EVAL_SERVICE_URL/sdk" \
#           | python -c "import sys,json; print(json.load(sys.stdin)['path'])")
#   curl -fsSL -H "Authorization: Bearer $EVAL_SERVICE_API_KEY" "${EVAL_SERVICE_URL}${WHEEL}" -o "/tmp/${WHEEL##*/}"
#   pip install --upgrade --force-reinstall --no-deps "/tmp/${WHEEL##*/}"
#   pip install httpx openai tqdm
import json, os, re
from simple_agentic_evals import EvalClient, ServiceError, react_agent, to_openai_tools
from openai import OpenAI
os.environ["EVAL_SERVICE_URL"] = "https://educator-marrow-cultural.ngrok-free.dev"
client = EvalClient(base_url=os.environ["EVAL_SERVICE_URL"], timeout=1800)
oai    = OpenAI()
MODEL  = "gpt-5"

# One EOG task, used by both modes below.
DATASET, BENCHMARK, DOMAIN = "evovling_agents", "eog", "itsm"
VERSION, SPLIT = 4, "train"
TASK_ID = "task_20260114_180601_485_99ba2325_2dc8b3ce"

In [5]:
PLAN = [
  {"id": "find_l1_group", "agent": "user_group",
   "tools": ["list_user_groups", "add_new_user_group", "update_user_group"],
   "input": "Use list_user_groups to retrieve all groups. Filter to the single group that is "
            "IT Support AND classified as a level 1 support team. Return the group's unique ID "
            "and name."},

  {"id": "select_and_escalate_oldest_incident", "agent": "incident",
   "tools": ["list_incidents", "get_incidents_assigned_to", "update_incident",
             "find_incident_by_id", "find_incident_by_number"],
   "input": "Using ${find_l1_group}, call get_incidents_assigned_to for that group. Keep "
            "incidents whose status is not resolved/closed, sort by created_on ascending and "
            "take the oldest. Call update_incident to set impact=high, urgency=high, "
            "priority=critical. Return incident number, opened_by and assigned_to."},

  {"id": "fetch_users_for_notifications", "agent": "user",
   "tools": ["get_user", "get_user_using_email", "get_user_using_name", "list_users"],
   "input": "Using ${select_and_escalate_oldest_incident}, call get_user for the reporter and "
            "the assignee. Return each user's id, full name and email."},

  {"id": "send_escalation_notifications", "agent": "notification",
   "tools": ["send_notification"],
   "input": "Using ${fetch_users_for_notifications} and "
            "${select_and_escalate_oldest_incident}, send two notifications via "
            "send_notification: one to the reporter and one to the assignee, each greeting the "
            "person by full name and naming the escalated incident."},
]
# Topological order; the real parser derives this from the <edge> block.
ORDER = [n["id"] for n in PLAN]

### Open the task and connect to the gym

`client.task(...)` provisions a **fresh database**. `task.mcp_session(server)` is the acting
surface.

In [7]:
task = client.task(DATASET, BENCHMARK, VERSION, TASK_ID, split=SPLIT, domain=DOMAIN)
task.__enter__()                       # kept open across the cells below

mcp        = task.mcp_session(task.mcp_servers[0])
all_tools  = mcp.list_tools()
by_name    = {t["name"]: t for t in all_tools}
print(f"{len(all_tools)} tools on {task.mcp_servers[0].name}")
print("task:", (task.user_prompt or ""), "...")

93 tools on gym-itsm-mcp
task: The group whose type is IT Support and that is classified as a level 1 support team has adopted a new rule that must be applied today: the oldest incident that is still unresolved and currently assigned to this group must be escalated by setting its impact, urgency, and priority to their highest available levels and by sending reports both to the user who reported it and to the user who is currently assigned to resolve it. The report for the user who originally reported this incident needs to have the subject Good News: Your Incident Has Been Escalated for Immediate Resolution. In the message, greet this reporter by their full name and congratulate them, explaining that their incident has been promoted and will now be handled as a top priority instead of waiting in the normal queue. Make it clear that the expectation is for their incident to be fully resolved no later than tomorrow. On the other hand, the report for the user who is currently assigned to r

### Run the plan, node by node

Instead of handing the task to a harness, we walk the DAG:
each node gets its own short-lived ReAct loop, its own system prompt, and **only its own tools**.

Nodes run strictly sequentially, they all mutate one shared database, so concurrent writes
would make the grade non-reproducible.

In [8]:
def run_node(node, upstream):
    """One subagent: its tools only, its instruction with ${refs} resolved."""
    instruction = re.sub(r"\$\{(\w+)\}",
                         lambda m: upstream.get(m.group(1), f"[{m.group(1)} unavailable]"),
                         node["input"])
    tools = to_openai_tools([by_name[t] for t in node["tools"] if t in by_name])
    messages = [
        {"role": "system", "content":
            f"You are the '{node['agent']}' specialist. Use ONLY your own tools. "
            f"Finish with a short report of what you did and the IDs you touched.\n\n"
            f"Overall task for context:\n{task.user_prompt or ''}"},
        {"role": "user", "content": instruction},
    ]
    for _ in range(12):                                  # per-node step cap
        msg = oai.chat.completions.create(
            model=MODEL, messages=messages, tools=tools).choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            return msg.content or ""
        for tc in msg.tool_calls:                        # ← the only service-side acting
            out = mcp.call_tool(tc.function.name, json.loads(tc.function.arguments or "{}"))
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out)})
    return "[step cap reached]"


upstream = {}
for node_id in ORDER:
    node = next(n for n in PLAN if n["id"] == node_id)
    upstream[node_id] = run_node(node, upstream)
    print(f"[{node_id:36}] {len(node['tools'])} tools -> {upstream[node_id]}...")

[find_l1_group                       ] 3 tools -> - Group ID: GROUP_001
- Name: Level 1 Support Team

Short report:
- Action: Called list_us...
[select_and_escalate_oldest_incident ] 5 tools -> Oldest unresolved incident in Level 1 Support Team (GROUP_001) escalated

Incident selecte...
[fetch_users_for_notifications       ] 4 tools -> Confirmation of escalation applied to oldest unresolved incident in Level 1 Support Team (...
[send_escalation_notifications       ] 1 tools -> Notifications dispatched successfully.

Reporter notification
- To: carlos.rodriguez@techc...


Close MCP first, then grade. `task.grade()` ignores everything the agents *said* and runs the
task's hidden SQL verifiers against the **final database state**.

In [9]:
mcp.close()
res = task.grade(keep_alive=True)

print(f"pass_rate={res.pass_rate:.2f}  ({res.n_passed}/{res.n_total})  success={res.overall_success}\n")
for v in res.per_verifier or []:
    print(f"  [{'PASS' if v['passed'] else 'FAIL'}] {v['name']:38} "
          f"expected={v['expected']} actual={v['actual']}")

task.__exit__(None, None, None)        # tear the session down

pass_rate=1.00  (3/3)  success=True

  [PASS] validate_incident_escalation_fields    expected=1 actual=1
  [PASS] validate_reporter_notification_escalated expected=1 actual=1
  [PASS] validate_assignee_notification_urgent  expected=1 actual=1


---
## Recap

**Every service call this pipeline makes on EOG — six:**

```python
client.benchmarks()                 # catalog / health
client.task_ids(...)                # locate a task's slice
client.task(...)                    # provision gym + fresh DB   -> Task
task.mcp_session(server)            # acting surface             -> MCPSession
  session.list_tools() / call_tool() / close()
task.grade(keep_alive=True)         # hidden SQL verifiers
```

We never call `react_agent`, `acp_codex_agent` or `task.evaluate()` in the real pipeline — those
run the agent on the service, which would leave us with no plan to score and no per-node trace.

**On ALE it inverts.** There is no MCP at all; the service runs Codex inside a Docker sandbox, so
we inject the plan into its orchestrator instead of executing it ourselves:

```python
task.start()
client._run_agent_job(session_id, {"agent": "codex", "prompt_suffix": <the plan>, ...})
client._get_bytes(f"/v1/sessions/{sid}/run_artifacts")
task.grade(keep_alive=True)
```

**What we do not use:** the evolving resource API. The agents and skills themselves are fully in
play — the roster constrains which subagents a plan may contain, and each agent's tool allowlist
is the `tools` list above — but they are read from a pinned `(domain, version)` rather than
fetched per stage.